# Week (2) Module A: Sampling Methods & the Central Limit Theorem
**Course:** Statistics and Machine Learning

---

### Module Contributor(s)
| Full Name | Role |
|---|---|
| `_______________` | `_______________` |
| `_______________` | `_______________` |

---

### About this Module
This module covers **Tasks 1, 2, and 3** of the Week (2) practical.
Your outputs will be merged with **Modules B, C, and D into the group's final notebook**.

**Dataset:** Pima Indians Diabetes → DOI: https://www.openml.org/d/37

| Feature | Description |
|---|---|
| `Pregnancies` | Number of pregnancies |
| `Glucose` | Plasma glucose concentration |
| `BloodPressure` | Diastolic blood pressure (mm Hg) |
| `SkinThickness` | Triceps skin fold thickness (mm) |
| `Insulin` | 2-hour serum insulin (mu U/ml) |
| `BMI` | Body mass index |
| `DiabetesPedigreeFunction` | Genetic risk score |
| `Age` | Age in years |
| `Outcome` | 0 = Non-diabetic, 1 = Diabetic |

---

**Background: Day 3 (Week 1) lecture recap:**

We rarely have access to an entire population, so we collect a **sample** and use it to draw conclusions. Three ideas from Day 3 are central to this module:

> - **Sampling methods:** How we choose the sample determines whether it represents the population. Random, systematic, stratified, and cluster sampling each offer different tradeoffs between cost, bias, and representativeness.
> - **Sampling bias:** A non-representative sample leads to inaccurate conclusions. Random sampling minimises bias by giving every individual an equal chance of selection.
> - **Central Limit Theorem (CLT):** For sufficiently large random samples (n ≥ 30), the distribution of the sample mean approaches a Normal distribution regardless of the population's shape. This underpins almost all of the inferential statistics in Module B.

---
> Cells marked `# TODO` require code. Cells marked `✍️` require a written response.

---
# Task 1: Sampling Methods
Before analysing any data, we must understand how it was collected. The sampling method determines whether the sample is representative of the population.

> **Goal:** Implement random, systematic, and stratified sampling on the Diabetes dataset and compare each sample's mean Glucose against the population mean.

In [1]:
# ── Task 1 Setup (do not modify) ──────────────────────────────────────────────

# pandas: tabular data manipulation (DataFrames, read_csv, groupby, etc.)
import pandas as pd

# numpy: numerical operations: arrays, random numbers, mathematical functions
import numpy as np

# matplotlib + seaborn: the standard Python plotting pair used throughout this course
import matplotlib.pyplot as plt
import seaborn as sns

# Apply a consistent visual style to every plot produced in this task.
# 'whitegrid' adds light horizontal gridlines; 'muted' uses softer, print-friendly colours.
sns.set_theme(style='whitegrid', palette='muted')

# Default figure dimensions (width × height in inches): individual plots override this.
plt.rcParams['figure.figsize'] = (10, 4)

# Load the Pima Indians Diabetes dataset from a public GitHub URL.
# Using a URL avoids local file dependencies, the notebook runs anywhere.
URL = 'https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv'
df = pd.read_csv(URL)

# Fix the random seed to get identical samples when running provided cells.
np.random.seed(42)

print(f'Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns')


Dataset loaded: 768 rows x 9 columns


In [2]:
# Provided: simple random sampling and population benchmark

# --- Population benchmark ---
# Before sampling, record the TRUE population mean so we can measure each sample's bias.
# 'Population' here means all 768 rows in the dataset, not literal all diabetic patients.
pop_mean_glucose = df['Glucose'].mean()
print(f'Population mean Glucose : {pop_mean_glucose:.2f} mg/dL  (this is our benchmark)')

# --- Simple Random Sampling ---
# .sample(n, random_state) draws n rows uniformly at random WITHOUT replacement.
# Every row has an equal probability of being chosen, minimising selection bias.
# Fixing random_state ensures the exact same 100 rows are chosen every run.
SAMPLE_SIZE = 100
random_sample = df.sample(n=SAMPLE_SIZE, random_state=42)

rand_mean = random_sample['Glucose'].mean()
print(f'Random sample mean      : {rand_mean:.2f} mg/dL')
print(f'Difference from pop mean: {abs(rand_mean - pop_mean_glucose):.2f} mg/dL')


Population mean Glucose : 120.89 mg/dL  (this is our benchmark)
Random sample mean      : 119.95 mg/dL
Difference from pop mean: 0.94 mg/dL


In [3]:
# Provided: systematic sampling, select every k-th row after a random start

# --- Systematic Sampling ---
# Step 1: Calculate the sampling interval k = N / n.
#         With 768 patients and n=100, we select approximately every 7th row.
N = len(df)                      # total number of rows in the dataset
k = N // SAMPLE_SIZE             # integer division, floor(768/100) = 7

# Step 2: Pick a random starting index between 0 and k-1.
#         Starting at a random point prevents always choosing the same first rows.
start = np.random.randint(0, k)

# Step 3: Select every k-th row beginning from 'start'.
#         np.arange produces index positions; .iloc retrieves those rows by position.
indices = np.arange(start, N, k)                   # e.g. [3, 10, 17, 24, ...]
systematic_sample = df.iloc[indices[:SAMPLE_SIZE]]  # cap at exactly SAMPLE_SIZE rows

sys_mean = systematic_sample['Glucose'].mean()
print(f'Systematic sample mean  : {sys_mean:.2f} mg/dL')
print(f'Difference from pop mean: {abs(sys_mean - pop_mean_glucose):.2f} mg/dL')


Systematic sample mean  : 117.93 mg/dL
Difference from pop mean: 2.96 mg/dL


In [ ]:
# TODO: Implement stratified sampling by DiabetesStatus (Diabetic / Non-Diabetic).
#
# Why stratified sampling?
# The dataset is imbalanced (~35% Diabetic, ~65% Non-Diabetic).
# Simple random sampling might over- or under-represent one class by chance.
# Stratified sampling draws proportionally from each group, guaranteeing correct representation.
#
# Steps:
#   1. Add a 'DiabetesStatus' column: map Outcome 0→'Non-Diabetic', 1→'Diabetic'.
#      (Use df['Outcome'].map({0: 'Non-Diabetic', 1: 'Diabetic'}))
#   2. Use .groupby('DiabetesStatus', group_keys=False).apply(
#          lambda g: g.sample(frac=0.13, random_state=42))
#      frac=0.13 draws ~13% of each group, giving roughly 100 rows total.
#   3. Store the result in a variable called stratified_sample.
#   4. Print: the total sample size, the count from each stratum, and the mean Glucose.
#   5. Print the difference from pop_mean_glucose.

# YOUR CODE HERE


In [ ]:
# TODO: Plot a bar chart comparing mean Glucose across all the three sampling methods.
#
# The chart should have four bars:
#   'Population'  |  'Random'  |  'Systematic'  |  'Stratified'
#
# Steps:
#   1. Create a dictionary mapping each label to its mean Glucose value.
#   2. Use pd.Series(means_dict).plot(kind='bar', ...) or plt.bar(...).
#   3. Add a horizontal dashed line at pop_mean_glucose as a visual reference.
#      (Use ax.axhline(pop_mean_glucose, linestyle='--', color='red', label='Population mean'))
#   4. Set the y-axis label to 'Mean Glucose (mg/dL)'.
#   5. Add a legend and a descriptive title.
#   6. Annotate each bar with its exact value (rounded to 2 dp).
#      (Hint: use ax.bar_label(bars, fmt='%.2f', padding=2))

# YOUR CODE HERE


### ✍️ Interpretation: Task 1

1. Which sampling method produced the mean closest to the population mean? Why might that be?
2. The dataset has roughly 65% Non-Diabetic and 35% Diabetic patients. How does stratified sampling handle this imbalance compared to random sampling?
3. In a real clinical study with 10,000 patients spread across 50 hospitals, which method would you use to select 500 patients quickly and why?

> *Your answer here:*

---
# Task 2: Sampling Bias & Sampling Distributions
Even a correctly chosen sampling method can produce a biased result by chance especially with small samples. The sampling distribution shows us how much sample means vary.

> **Goal:** Demonstrate sampling bias by comparing biased vs unbiased sampling, then simulate a sampling distribution and observe how it narrows as sample size grows.

In [4]:
# ── Task 2 Setup (do not modify) ──────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 4)

URL = 'https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv'
df = pd.read_csv(URL)
df['DiabetesStatus'] = df['Outcome'].map({0: 'Non-Diabetic', 1: 'Diabetic'})

np.random.seed(42)  # RANDOM_STATE=42 used across all modules for consistency

# Extract Glucose as a NumPy array for efficient repeated sampling in the simulation below.
# We work with the raw column (including zeros) to show the real population distribution.
glucose_pop = df['Glucose'].values

print('Ready')


Ready


In [5]:
# Provided: demonstrate sampling bias by restricting the sampling frame

# --- Unbiased sample ---
# Drawn from the entire dataset, every patient has an equal chance of inclusion.
unbiased = df.sample(n=50, random_state=42)
unbiased_mean = unbiased['Glucose'].mean()

# --- Biased sample ---
# Only Diabetic patients are in the sampling frame.
# This simulates a real bias: e.g. only patients who visited a diabetes clinic were surveyed.
# The resulting mean will be systematically higher than the true population mean.
diabetic_only = df[df['Outcome'] == 1]
biased = diabetic_only.sample(n=50, random_state=42)
biased_mean = biased['Glucose'].mean()

pop_mean = df['Glucose'].mean()

print(f'True population mean      : {pop_mean:.2f} mg/dL')
print(f'Unbiased sample mean      : {unbiased_mean:.2f} mg/dL  (error: {abs(unbiased_mean-pop_mean):.2f})')
print(f'Biased sample mean        : {biased_mean:.2f} mg/dL  (error: {abs(biased_mean-pop_mean):.2f})')
print("==================================================================================================")
print('The biased sample overestimates the population mean because it only includes')
print('patients who already have diabetes (a classic sampling frame error).')


True population mean      : 120.89 mg/dL
Unbiased sample mean      : 123.04 mg/dL  (error: 2.15)
Biased sample mean        : 138.36 mg/dL  (error: 17.47)
The biased sample overestimates the population mean because it only includes
patients who already have diabetes (a classic sampling frame error).


In [ ]:
# Provided: simulate a sampling distribution for n=10 to visualise spread

# --- What is a sampling distribution? ---
# If we repeated the sampling process many times, we would get a slightly different
# sample mean each time. The distribution of all those means is the 'sampling distribution'.
# As n increases, this distribution becomes narrower and more Normal (CLT, see Task 3).

N_SIMULATIONS = 2000   # number of times we repeat the sampling process
n = 10                 # sample size per draw, deliberately small to show high spread

# List comprehension: draw a random sample of size n, compute its mean, repeat 2000 times.
sample_means_10 = [
    np.random.choice(glucose_pop, size=n, replace=True).mean()
    for _ in range(N_SIMULATIONS)
]

fig, ax = plt.subplots(figsize=(8, 4))

# Plot the distribution of sample means as a histogram.
# kde=True overlays a smooth density curve so the shape is easier to read.
ax.hist(sample_means_10, bins=40, color='steelblue', edgecolor='white',
        alpha=0.8, density=True, label='Sample means (n=10)')

# Mark the true population mean with a vertical line for comparison.
ax.axvline(pop_mean, color='red', linestyle='--', linewidth=2,
           label=f'Population mean = {pop_mean:.1f}')

ax.set_title('Sampling Distribution of the Mean (n=10, 2000 simulations)')
ax.set_xlabel('Sample Mean Glucose (mg/dL)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean of sample means : {np.mean(sample_means_10):.2f}  (should be close to {pop_mean:.2f})')
print(f'Std of sample means  : {np.std(sample_means_10):.2f}  (this is the Standard Error of the Mean)')


In [ ]:
# TODO: Repeat the simulation for n=30 and n=100, then plot all three side-by-side.
#
# Steps:
#   1. Generate sample_means_30 and sample_means_100 using the same pattern as above.
#      Use N_SIMULATIONS=2000 and np.random.choice(glucose_pop, size=n, replace=True).
#   2. Create a figure with 3 subplots side-by-side:
#      plt.subplots(1, 3, figsize=(15, 4))
#   3. In each subplot:
#      - Plot the histogram of sample means (bins=40, density=True).
#      - Add a vertical red dashed line at pop_mean.
#      - Set the title to: f'n = {n}  |  SE = {np.std(means):.2f}'
#      - Label x-axis 'Sample Mean Glucose (mg/dL)' and y-axis 'Density'.
#   4. Add a main title: 'Sampling Distribution Narrows as n Increases'.
#
# After plotting, print a summary table:
#   n  |  Mean of sample means  |  Std of sample means (= SE)
# for all three values of n.

# YOUR CODE HERE


### ✍️ Interpretation: Task 2

1. The biased sample mean was higher than the population mean. Explain in plain English WHY sampling only from Diabetic patients causes this error.
2. As sample size grew from 10 → 30 → 100, what happened to the spread (standard deviation) of the sampling distribution? What formula governs this relationship?
3. All three sampling distributions are centred near the population mean even though individual samples vary. What does this tell us about random sampling as a strategy?

> *Your answer here:*

---
# Task 3: Central Limit Theorem (CLT) Simulation
The CLT is one of the most powerful results in all of statistics. It explains why the Normal distribution appears so frequently in practice — even when the underlying data is not Normal.

> **Goal:** Prove the CLT empirically by showing that sample means become approximately Normal as n increases, regardless of the population's original shape.

In [ ]:
# ── Task 3 Setup (do not modify) ──────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats   # scipy.stats: probability distributions, hypothesis tests, etc.

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 4)

URL = 'https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv'
df = pd.read_csv(URL)

np.random.seed(42)  # RANDOM_STATE=42 used across all modules

# Use Insulin, it is heavily right-skewed (many zeros/low values, long right tail).
# This makes the CLT result visually dramatic: the population is very non-Normal,
# yet the sample means converge to a Normal distribution.
insulin_pop = df['Insulin'].values   # shape: (768,)

print(f'Insulin population: n={len(insulin_pop)}, mean={insulin_pop.mean():.2f}, '
      f'skewness={stats.skew(insulin_pop):.2f}')
print('Ready')


In [ ]:
# Provided: plot the population distribution to confirm it is highly non-Normal

fig, ax = plt.subplots(figsize=(7, 4))

# sns.histplot with kde=True plots the histogram AND a smooth density curve.
# stat='density' scales the y-axis to probability density (area under the bar = 1),
# so the KDE and the histogram are on the same scale.
sns.histplot(insulin_pop, bins=35, kde=True, ax=ax,
             color='coral', stat='density')

ax.axvline(insulin_pop.mean(), color='navy', linestyle='--', linewidth=2,
           label=f'Mean = {insulin_pop.mean():.1f}')

ax.set_title('Population Distribution: Insulin (heavily right-skewed: NOT Normal)')
ax.set_xlabel('Insulin (mu U/ml)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

print('Skewness > 0 confirms right skew. The CLT will make sample means Normal despite this.')


In [ ]:
# Provided: CLT simulation for n=5 with theoretical Normal overlay

# --- CLT simulation ---
# We repeat 3000 times:
#   draw a sample of size n → compute its mean → record the mean.
# The histogram of all 3000 means is the sampling distribution.
# The CLT predicts this should look Normal with:
#   mean  = population mean (mu)
#   std   = population std / sqrt(n)   ← this is called the Standard Error (SE)

N_SIMS = 3000
n_demo = 5   # small sample: CLT is not yet fully active

means_n5 = np.array([
    np.random.choice(insulin_pop, size=n_demo, replace=True).mean()
    for _ in range(N_SIMS)
])

# Compute the theoretical Normal the CLT predicts.
mu_theory  = insulin_pop.mean()                   # CLT prediction: mean of sample means = population mean
se_theory  = insulin_pop.std() / np.sqrt(n_demo)  # CLT prediction: SE = sigma / sqrt(n)

# Create a range of x values for plotting the theoretical Normal curve.
x_range = np.linspace(means_n5.min(), means_n5.max(), 300)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(means_n5, bins=50, density=True, color='coral',
        edgecolor='white', alpha=0.8, label='Simulated sample means')

# Overlay the Normal PDF predicted by the CLT.
# stats.norm.pdf(x, loc=mu, scale=sigma) computes the Normal probability density at x.
ax.plot(x_range, stats.norm.pdf(x_range, mu_theory, se_theory),
        color='navy', linewidth=2.5, label='CLT: predicted Normal')

ax.set_title(f'Sampling Distribution of Mean Insulin  (n={n_demo}, {N_SIMS} simulations)\n'
             f'still skewed at small n')
ax.set_xlabel('Sample Mean Insulin (mu U/ml)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# TODO: Repeat the CLT simulation for n=30 and n=100.
#       Plot them alongside the n=5 result so the convergence is visible.
#
# Steps:
#   1. Generate means_n30 and means_n100 using the same loop pattern as above.
#      Use N_SIMS=3000 and np.random.choice(insulin_pop, size=n, replace=True).
#
#   2. Create ONE figure with 3 subplots: plt.subplots(1, 3, figsize=(16, 4)).
#      For each subplot (n=5, n=30, n=100):
#        a. Plot the histogram of sample means (density=True, bins=50, edgecolor='white').
#        b. Overlay the CLT Normal curve using stats.norm.pdf().
#           Remember: SE = insulin_pop.std() / np.sqrt(n).
#        c. Title: f'n = {n}  |  Skewness = {stats.skew(means):.3f}'
#        d. Label axes consistently.
#
#   3. Add a suptitle: 'Central Limit Theorem: Sample Means Converge to Normal as n Increases'.
#
#   4. After the plot, print a table with columns:
#      n | Empirical Mean | Empirical SE | Theoretical SE | Skewness
#      for all three sample sizes.
#      (Theoretical SE = insulin_pop.std() / np.sqrt(n))

# YOUR CODE HERE


In [ ]:
# TODO: Plot skewness vs sample size to make the CLT convergence quantitative.
#
# Steps:
#   1. Compute stats.skew() for means_n5, means_n30, and means_n100.
#   2. Also compute the skewness of the raw insulin_pop for comparison.
#   3. Create a line plot: x-axis = [5, 30, 100], y-axis = skewness of sample means.
#      Add a horizontal dashed line at y=0 (the skewness of a perfect Normal).
#   4. Mark the population skewness as a horizontal line of a different colour.
#   5. Title: 'Skewness of Sample Means Approaches 0 as n Grows (CLT)'.
#   6. Add appropriate axis labels and a legend.

# YOUR CODE HERE


### ✍️ Interpretation: Task 3

1. At which sample size (5, 30, or 100) did the sampling distribution look most Normal? How does the skewness value confirm your visual observation?
2. The Standard Error (SE = σ/√n) tells us how much sample means vary. What happens to SE when n is multiplied by 4? Why is this relationship important for study design?
3. The Insulin population is heavily right-skewed, yet the CLT still applies. Give one example from the lecture where this property is practically useful (e.g., in medicine or ML).

> *Your answer here:*

---
## 📋 Module A: Handoff Summary

Complete this table before merging with your group. It helps teammates understand your findings quickly.

| Item | Your finding |
|---|---|
| Best sampling method (smallest bias) | ??? |
| Random sample mean Glucose (n=100) | ??? mg/dL |
| Stratified sample mean Glucose (n≈100) | ??? mg/dL |
| Biased sample mean Glucose (Diabetic only) | ??? mg/dL |
| SE of sample means at n=10 | ??? |
| SE of sample means at n=100 | ??? |
| Minimum n where CLT looked Normal (Insulin) | ??? |
| Skewness of sample means at n=100 | ??? |

> Fill in the table above before handing off to the group.